# Notebook 03 — Audit Macro Vintage Data

## Goal
Inspect the raw FRED and ALFRED data. Understand what vintage data means, check data quality,
and create an approved macro candidate file for feature engineering.

---

## What is vintage data and why does it matter?

**Vintage data** captures the value of a series as it was *released at a specific point in time*, before any subsequent revisions.

For example, PAYEMS for 2010-03 may have been first reported as 162,000K jobs in April 2010,
then revised to 162,500K in May 2010, and again revised to 163,100K in the annual benchmark revision.
A model trained with the *final* revised value would be using data that was not available at forecast time.
This is **look-ahead bias** — a form of data leakage.

**ALFRED fields:**
- `realtime_start`: the date from which this vintage became the official release
- `realtime_end`: the date until which this vintage was the official release
- A row `(PAYEMS, 2010-03-01, value=162000, realtime_start=2010-04-02, realtime_end=2010-05-06)` means:
  from April 2 to May 6, 2010, the official value for March 2010 PAYEMS was 162,000K.

**Revised fallback**: if no vintage is found for a forecast date, use the closest available vintage.

---

## What can go wrong
- ALFRED does not have complete vintage coverage for all series before 1991
- ICSA is weekly — it needs aggregation to monthly before use
- JTSJOL starts in December 2000 — forecasts before that date will have missing values
- COVID months (2020-03 to 2020-05) will appear as extreme outliers in all series

---

## What you must inspect
- PAYEMS level and MoM change plots — do they look right?
- Are extreme values limited to COVID?
- Is JTSJOL missing before 2001? (expected)
- Are vintage revisions large for PAYEMS? (annual benchmark revisions are large)

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

with open(REPO_DIR / 'configs' / 'base.yaml') as f:
    base_cfg = yaml.safe_load(f)

RAW_FRED_PATH = DRIVE_ROOT / 'data' / 'raw' / 'fred' / 'fred_current.parquet'
RAW_ALFRED_PATH = DRIVE_ROOT / 'data' / 'raw' / 'alfred' / 'alfred_vintage.parquet'

# Load approval check
approval_01 = DRIVE_ROOT / 'approvals' / '01_raw_fred_alfred_approved.json'
if not approval_01.exists():
    raise FileNotFoundError('STOP: Notebook 01 has not been approved. Run and approve NB 01 first.')
with open(approval_01) as f:
    ap = json.load(f)
assert ap['approved'], 'STOP: Notebook 01 was not approved.'
print('Notebook 01 approval confirmed.')

In [ ]:
fred_df = pd.read_parquet(RAW_FRED_PATH)
alfred_df = pd.read_parquet(RAW_ALFRED_PATH)

print(f'FRED current: {fred_df.shape}')
print(f'ALFRED vintage: {alfred_df.shape}')
print()
print('FRED columns:', list(fred_df.columns))
print('ALFRED columns:', list(alfred_df.columns))

## Assertion functions

In [ ]:
def assert_required_columns(df, required_columns, name):
    missing = [c for c in required_columns if c not in df.columns]
    assert not missing, f'{name}: Missing required columns: {missing}'
    print(f'  [{name}] Required columns present: {required_columns}')

def assert_no_nulls(df, columns, name):
    for col in columns:
        n = df[col].isna().sum()
        assert n == 0, f'{name}: Column "{col}" has {n} null values'
    print(f'  [{name}] No nulls in: {columns}')

def assert_numeric_columns(df, columns, name):
    for col in columns:
        assert pd.api.types.is_numeric_dtype(df[col]), f'{name}: Column "{col}" is not numeric'
    print(f'  [{name}] Numeric columns confirmed: {columns}')

def assert_monthly_dates(df, date_col, name):
    dates = pd.to_datetime(df[date_col])
    days = dates.dt.day.unique()
    assert set(days).issubset({1}), f'{name}: observation_date is not all day=1 (monthly). Days found: {days}'
    print(f'  [{name}] Monthly dates confirmed (all day=1)')

print('Assertion functions defined.')

In [ ]:
print('Running assertions on FRED current data...')
assert_required_columns(fred_df, ['series_id', 'observation_date', 'value'], 'FRED')
assert_numeric_columns(fred_df, ['value'], 'FRED')

print('Running assertions on ALFRED vintage data...')
assert_required_columns(alfred_df, ['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'value'], 'ALFRED')
assert_numeric_columns(alfred_df, ['value'], 'ALFRED')

print('\nAll assertions passed.')

## Inspect PAYEMS — the target variable

In [ ]:
payems = fred_df[fred_df['series_id'] == 'PAYEMS'].copy()
payems = payems.sort_values('observation_date')
payems['mom_change'] = payems['value'].diff()

print('PAYEMS summary:')
print(f'  Rows: {len(payems)}')
print(f'  Date range: {payems["observation_date"].min().date()} to {payems["observation_date"].max().date()}')
print(f'  Value range: {payems["value"].min():.0f}K to {payems["value"].max():.0f}K')
print(f'  MoM change range: {payems["mom_change"].min():.0f}K to {payems["mom_change"].max():.0f}K')
print(f'  Missing values: {payems["value"].isna().sum()}')
print()
print('Largest MoM drops (top 5):')
print(payems.nsmallest(5, 'mom_change')[['observation_date', 'value', 'mom_change']].to_string(index=False))
print()
print('Largest MoM gains (top 5):')
print(payems.nlargest(5, 'mom_change')[['observation_date', 'value', 'mom_change']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(payems['observation_date'], payems['value'] / 1000, color='steelblue')
axes[0].set_title('PAYEMS — Total Nonfarm Payrolls (millions)')
axes[0].set_ylabel('Thousands (millions shown)')
axes[0].grid(True, alpha=0.3)

colors = ['red' if v < 0 else 'steelblue' for v in payems['mom_change']]
axes[1].bar(payems['observation_date'], payems['mom_change'], color=colors, width=25, alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('PAYEMS — Month-on-Month Change (thousands)')
axes[1].set_ylabel('Thousands')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'outputs' / 'figures' / 'payems_level_and_change.png'), dpi=100)
plt.show()

In [ ]:
plot_series = ['UNRATE', 'ICSA', 'JTSJOL']
fig, axes = plt.subplots(len(plot_series), 1, figsize=(14, 10))

for i, sid in enumerate(plot_series):
    sub = fred_df[fred_df['series_id'] == sid].sort_values('observation_date')
    axes[i].plot(sub['observation_date'], sub['value'], label=sid)
    axes[i].set_title(f'{sid}')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'outputs' / 'figures' / 'macro_series_overview.png'), dpi=100)
plt.show()

## Inspect ALFRED vintage coverage for PAYEMS

In [ ]:
payems_alfred = alfred_df[alfred_df['series_id'] == 'PAYEMS'].copy()
payems_alfred = payems_alfred.sort_values(['observation_date', 'realtime_start'])

print(f'PAYEMS in ALFRED: {len(payems_alfred)} vintage rows')
print(f'Unique observation dates: {payems_alfred["observation_date"].nunique()}')
print(f'realtime_start range: {payems_alfred["realtime_start"].min().date()} to {payems_alfred["realtime_start"].max().date()}')
print()

# Show how many vintages exist per observation date
vintages_per_date = payems_alfred.groupby('observation_date').size().describe()
print('Vintages per observation date:')
print(vintages_per_date.to_string())
print()

# Show an example: how many times was 2010-01 revised?
example = payems_alfred[payems_alfred['observation_date'] == '2010-01-01']
print('Example: PAYEMS 2010-01 revisions:')
print(example[['observation_date', 'realtime_start', 'realtime_end', 'value']].to_string(index=False))

## Create audited macro candidate file

In [ ]:
# For the audited file, normalize observation_date to first of month
# and ensure ALFRED vintage fields are present

macro_audited = alfred_df.copy()
macro_audited['observation_date'] = pd.to_datetime(macro_audited['observation_date'])
# Normalize to first of month
macro_audited['observation_date'] = macro_audited['observation_date'].dt.to_period('M').dt.to_timestamp()

print('Macro vintage audited shape:', macro_audited.shape)
print('Columns:', list(macro_audited.columns))
print()
print('Series in audited file:')
print(macro_audited['series_id'].value_counts().to_string())
print()
print('Missing values per column:')
print(macro_audited.isnull().sum().to_string())

In [ ]:
# Data quality report
dq_rows = []
for sid in macro_audited['series_id'].unique():
    sub = macro_audited[macro_audited['series_id'] == sid]
    dq_rows.append({
        'series_id': sid,
        'vintage_rows': len(sub),
        'unique_dates': sub['observation_date'].nunique(),
        'min_date': sub['observation_date'].min(),
        'max_date': sub['observation_date'].max(),
        'null_value_pct': sub['value'].isna().mean(),
    })

dq_report = pd.DataFrame(dq_rows)
print(dq_report.to_string(index=False))

dq_path = DRIVE_ROOT / 'outputs' / 'data_quality' / 'macro_vintage_audit_report.csv'
dq_report.to_csv(dq_path, index=False)
print(f'\nReport saved: {dq_path}')

## Manual Approval Gate

**Before approving, verify:**
1. PAYEMS level plot looks correct (upward trend with COVID dip in 2020)
2. MoM change plot shows extreme COVID drop (-20M+) and recovery
3. UNRATE, ICSA, JTSJOL plots look reasonable
4. Vintage fields are present for all series
5. No series has >5% missing values

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
out_path = DRIVE_ROOT / 'data' / 'interim' / 'macro' / 'macro_vintage_audited.parquet'
macro_audited.to_parquet(out_path, index=False)
print(f'Audited macro vintage saved: {out_path}')

approval = {
    'approved': True,
    'notebook': '03_audit_macro_vintage.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(RAW_FRED_PATH), str(RAW_ALFRED_PATH)],
    'output_artifacts': [str(out_path), str(dq_path)],
    'row_counts': {'macro_vintage_audited': int(len(macro_audited))},
    'warnings': [],
    'notes': ''
}
ap_path = DRIVE_ROOT / 'approvals' / '03_macro_vintage_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 03 complete. Proceed to 04_audit_and_clean_news.ipynb')